# Lesson 10: Validating Model-Graded Evals for Subjective Behavior

In lessons 1–9 we learned how to grade model outputs using code checks and model-graded evaluators. Those techniques work well for objective tasks — counting legs, classifying topics, summarizing text.

But what about **subjective** behavior — qualities like guidance style, tone, or pedagogical discipline? In this lesson we use **AI tutoring** as our example domain and build a complete, rigorous evaluation pipeline:

1. Define a **research-grounded behavioral taxonomy**
2. Add a **deterministic check** as a cheap first gate
3. Build and **validate a frozen rubric judge** against human-annotated golden labels
4. Run an **actor-critic optimization loop** to produce an improved prompt
5. **Compare baseline vs optimized** on held-out test data

> **Note:** This lesson uses the Anthropic API directly — no external eval framework required. A clean run makes about 20 API calls. With Claude Haiku this should cost only a few cents, though exact cost depends on current model pricing and generated output length.

## Why model-graded evals alone are not enough for subjective behavior

In lesson 9 we wrote a custom model-graded evaluator and trusted its judgment. For subjective tasks, that trust needs to be earned:

- A **generic evaluator** has no grounding in what *specifically* good tutoring looks like
- Without **calibration examples**, the evaluator applies inconsistent standards across runs
- Without a **frozen prompt**, the evaluator drifts when you iterate on data
- Without **validation against human labels**, you cannot know whether the evaluator is actually objective

This lesson addresses all four problems. The goal is not to claim that tutoring quality has a single universal ground truth. Instead, we define an explicit rubric, calibrate it against human labels, and freeze the judge — so that from that point on, different prompt variants can be compared under the same measurement procedure. **Subjective at the source, objective within the protocol.**

## Setup

```bash
pip install anthropic pydantic python-dotenv
```

Set your API key in a `.env` file:

```
ANTHROPIC_API_KEY=your_key_here
```

In [23]:
from anthropic import Anthropic
from dotenv import load_dotenv
from pydantic import BaseModel
import json

load_dotenv()
client = Anthropic()

ACTOR_MODEL     = 'claude-haiku-4-5-20251001'
JUDGE_MODEL     = 'claude-haiku-4-5-20251001'
OPTIMIZER_MODEL = 'claude-haiku-4-5-20251001'

## Our datasets

> **Important:** The examples below are **synthetic and created for this lesson**. They illustrate the data structures you need in a real project. The companion repository uses real human-annotated data from [MRBench](https://github.com/MRBench/MRBench), a benchmark of tutoring responses with expert labels. You must download MRBench separately to run the full pipeline.

A rigorous evaluation pipeline requires **three strictly separate datasets**, each with a distinct role:

| Dataset | Purpose | Has golden labels? |
|---|---|---|
| Judge validation | Verify the judge agrees with humans | Yes — human-annotated |
| Optimization / train | Run the actor-critic loop | No |
| Evaluation / test | Compare baseline vs optimized | No |

Keeping these three sets strictly separate prevents data leakage and ensures that evaluation results are not inflated by examples the optimizer has already seen.

In [24]:
# Judge validation data: pre-written tutor responses + human-annotated golden labels.
# In the companion repository these come from MRBench expert annotations.
judge_val_data = [
    {
        'id': 'val_001',
        'student_question': 'I think 18 divided by 2 equals 30. Can you check that for me?',
        'tutor_response': '18 divided by 2 is 9, not 30. You may have confused multiplication with division.',
        'golden_labels': {
            'mistake_identification':       True,
            'mistake_location':             False,
            'answer_revealing_appropriate': False,  # stated 9 directly
            'providing_guidance':           False,
            'actionability':                False,
            'coherence':                    True,
            'tutor_tone':                   True,
            'human_likeness':               True,
        }
    },
    {
        'id': 'val_002',
        'student_question': 'I said a pentagon has 6 sides. My teacher marked it wrong but I am not sure why.',
        'tutor_response': "Great question! The prefix 'penta' is a clue — what other words do you know that start with 'penta'? Once you work that out, you will know exactly how many sides a pentagon has.",
        'golden_labels': {
            'mistake_identification':       True,
            'mistake_location':             True,
            'answer_revealing_appropriate': True,
            'providing_guidance':           True,
            'actionability':                True,
            'coherence':                    True,
            'tutor_tone':                   True,
            'human_likeness':               True,
        }
    },
    {
        'id': 'val_003',
        'student_question': 'I think the boiling point of water is 90 degrees Celsius at sea level. Is that right?',
        'tutor_response': 'You are close! The boiling point of water at sea level is actually 100 degrees Celsius — just 10 degrees off from your answer.',
        'golden_labels': {
            'mistake_identification':       True,
            'mistake_location':             False,
            'answer_revealing_appropriate': False,  # stated 100 directly
            'providing_guidance':           False,
            'actionability':                False,
            'coherence':                    True,
            'tutor_tone':                   True,
            'human_likeness':               True,
        }
    },
]

print(f'Judge validation examples: {len(judge_val_data)}')

Judge validation examples: 3


In [25]:
# Optimization data: student questions only — no golden labels needed.
# The actor-critic loop uses these to generate and score responses during prompt improvement.
actor_train_data = [
    {'id': 'train_001', 'student_question': 'For the equation 2x + 4 = 10, I got x = 7. Is that correct?'},
    {'id': 'train_002', 'student_question': 'I calculated the derivative of x squared as just x. Did I do that right?'},
    {'id': 'train_003', 'student_question': 'I believe the Great Wall of China is visible from space with the naked eye. Is that true?'},
]

# Evaluation data: held-out test set, never seen during optimization.
# Both baseline and optimized prompts are evaluated here for a fair comparison.
actor_test_data = [
    {'id': 'test_001', 'student_question': 'I think friction always slows things down. Is that right?'},
    {'id': 'test_002', 'student_question': 'I said photosynthesis happens in the mitochondria. My teacher said I was wrong.'},
]

print(f'Optimization examples: {len(actor_train_data)}')
print(f'Test examples:         {len(actor_test_data)}')

Optimization examples: 3
Test examples:         2


## Our initial tutor prompt (the actor)

We start with the most natural prompt someone would write for a tutoring AI. This is our **baseline** — a short, generic instruction to be helpful. The central question of this lesson: *does this prompt actually produce good tutoring behavior?*

In [26]:
BASELINE_ACTOR_PROMPT = 'You are an expert, helpful academic AI tutor. Please respond to the student.'

def get_tutor_response(student_question, system_prompt=BASELINE_ACTOR_PROMPT):
    response = client.messages.create(
        model=ACTOR_MODEL,
        max_tokens=400,
        system=system_prompt,
        messages=[{'role': 'user', 'content': student_question}]
    )
    return response.content[0].text

Let's preview what the baseline prompt produces on one of our training questions:

In [27]:
preview_response = get_tutor_response(actor_train_data[0]['student_question'])
print('Student:', actor_train_data[0]['student_question'])
print()
print('Tutor:', preview_response)

Student: For the equation 2x + 4 = 10, I got x = 7. Is that correct?

Tutor: No, that's not correct. Let me walk you through the solution:

**Starting equation:** 2x + 4 = 10

**Step 1:** Subtract 4 from both sides
- 2x + 4 - 4 = 10 - 4
- 2x = 6

**Step 2:** Divide both sides by 2
- 2x ÷ 2 = 6 ÷ 2
- **x = 3**

**Check your answer:** Let's substitute x = 3 back into the original equation:
- 2(3) + 4 = 6 + 4 = 10 ✓

So **x = 3** is the correct answer.

Where did you get x = 7? If you can show me your work, I can help you identify where the mistake occurred!


Even on this first example we can suspect a problem: did the tutor reveal the answer directly? A model that tells the student '2x + 4 = 10 means 2x = 6, so x = 3' has removed all cognitive challenge from the interaction. This is *helpful* in a narrow sense but pedagogically counterproductive.

We cannot diagnose this reliably by reading two responses. We need to measure it systematically. Let's build the evaluator.

## Step 1: Define a research-grounded behavioral taxonomy

Instead of a single score, we evaluate tutoring across **8 dimensions**. This is not an arbitrary choice — each dimension maps to a concept established in educational research:

| Dimension | Research grounding |
|---|---|
| `mistake_identification` | Formative assessment theory (Black & Wiliam, 1998): distinguishing *whether* an error exists |
| `mistake_location` | Same: distinguishing *where* in the reasoning the error occurred |
| `answer_revealing_appropriate` | Vygotsky's Zone of Proximal Development and the Socratic method: the tutor should scaffold discovery, not remove the cognitive challenge |
| `providing_guidance` | Feedback quality research (Hattie & Timperley, 2007): effective feedback must point toward a corrective path |
| `actionability` | Same: feedback must give something concrete to try next |
| `coherence` | General communication quality in Intelligent Tutoring Systems (ITS) research |
| `tutor_tone` | Motivation and engagement research: encouraging tone correlates with persistence |
| `human_likeness` | HCI and dialogue research: natural responses support engagement |

These dimensions are inspired by recurring themes in formative assessment, feedback quality, and tutoring research. Grounding a rubric in prior work helps explain *why* each dimension matters and makes it easier to communicate evaluation criteria to domain experts.

> **Analogy:** In NLP, emotion classifiers like GoEmotions ground their 27 categories in psychology research. Here we build an analogous taxonomy for tutoring quality — with the same goal of turning subjective human judgments into objective, measurable signals.

We'll encode this taxonomy as a Pydantic schema. This ensures the judge must produce **exactly** this structure — no missing dimensions, no ambiguous fields.

In [28]:
class DimensionJudgment(BaseModel):
    passed: bool
    reason: str

class TutorEvaluation(BaseModel):
    mistake_identification:       DimensionJudgment
    mistake_location:             DimensionJudgment
    answer_revealing_appropriate: DimensionJudgment
    providing_guidance:           DimensionJudgment
    actionability:                DimensionJudgment
    coherence:                    DimensionJudgment
    tutor_tone:                   DimensionJudgment
    human_likeness:               DimensionJudgment

DIMENSIONS = list(TutorEvaluation.model_fields.keys())
print('Dimensions:', DIMENSIONS)

Dimensions: ['mistake_identification', 'mistake_location', 'answer_revealing_appropriate', 'providing_guidance', 'actionability', 'coherence', 'tutor_tone', 'human_likeness']


## Step 2: Deterministic check as a first gate

Before invoking the model-graded judge (which costs API calls), we run a **zero-cost deterministic check**: does the response contain obvious direct-answer phrases?

This is the same principle as lessons 3 and 4 — use code-graded evaluation for everything you can check mechanically, and reserve the model-graded judge for what genuinely requires language understanding. The two layers complement each other:

- Deterministic check: fast, free, catches obvious violations
- Model-graded judge: slower, costs tokens, handles nuance

In [29]:
DIRECT_ANSWER_PHRASES = [
    'the answer is',
    'the correct answer',
    'that is incorrect, it is',
    'actually equals',
    'it should be',
]

def deterministic_check(response: str) -> bool:
    """Returns True if no obvious direct-answer pattern is detected."""
    response_lower = response.lower()
    return not any(phrase in response_lower for phrase in DIRECT_ANSWER_PHRASES)

# Sanity check
print(deterministic_check('Let me help you think through this step by step.'))  # True
print(deterministic_check('That is incorrect, it is 9.'))                       # False

True
False


## Step 3: Build the frozen rubric judge

### Why do we need to build a judge at all?

In some evaluation tasks, a pre-trained classifier already exists. For example, if you want to classify the emotional tone of text, you can use the **GoEmotions** model — a fine-tuned classifier trained on 27 emotion categories from 58,000 annotated Reddit comments. You use it directly; no building required.

For **tutoring quality**, no such classifier exists. Our 8 dimensions are domain-specific and require understanding of pedagogical context — something no general-purpose classifier can handle. We'll therefore build our own judge using Claude, calibrated against human-annotated examples.

This is a common situation in applied NLP: when no off-the-shelf model covers your evaluation task, you ground a general-purpose LLM in your specific criteria through careful prompt design.

### What does "frozen" mean?

A frozen judge is a model-graded evaluator whose **system prompt is fixed before the optimization loop starts** and never changes during it. This is essential for two reasons:

1. **Comparability**: if the judge's criteria change between iterations, your metrics are not comparable across runs
2. **Auditability**: anyone can read exactly what criteria the judge is applying — it is a transparent, inspectable artifact in source control

### Calibrating the judge with few-shot prompting

Left to its own judgment, a model applies generic, potentially inconsistent standards. The industry solution is **few-shot prompting** (also called in-context learning):

Inject 2–5 representative *gold-labeled examples* directly into the system prompt. Each example shows:
- A student question
- A tutor response
- The correct human judgment for each dimension

The model then uses these anchors to calibrate its own scoring. This is the perfect middle ground between a raw prompt (no calibration) and fine-tuning (expensive, requires thousands of examples). Three to five high-quality examples are sufficient to significantly reduce variance in the judge's output.

> **Best practice:** Choose calibration examples that are representative of edge cases, not just easy ones. Include at least one example where a key dimension passes and one where it fails.

### Enforcing structure with forced tool use

Few-shot prompting calibrates *what the judge scores*. A second technique enforces *how the judge reports those scores*: **forced tool use**.

By setting `tool_choice={"type": "tool", "name": "..."}` in the API call, you instruct the model that it *must* invoke the named tool — it cannot produce a free-form text response. This has two concrete effects:

1. **Machine-parseable output**: every judge call returns a JSON object with exactly the required fields — no parsing heuristics, no truncation, no silently missing dimensions
2. **Schema validation via Pydantic**: the JSON is immediately validated against the `TutorEvaluation` schema; if the model hallucinates a field name or returns the wrong type, the code raises an error before a bad score can propagate

Together, few-shot prompting and forced tool use close the two most common failure modes of LLM-as-judge systems: **inconsistent scoring criteria** and **inconsistent output format**. In production, you should never rely on one without the other.

> **Best practice:** Always pair `tool_choice` with a schema validator (Pydantic, JSON Schema, or equivalent). The API guarantees the model calls the tool; the validator guarantees the output matches your schema. Each layer catches a different class of failure.

In [30]:
# This prompt is frozen: once set, it does not change during the optimization loop.
# The two calibration examples are drawn from human-annotated data (golden labels).
JUDGE_SYSTEM_PROMPT = (
    'You are an expert evaluator of AI tutor responses. '
    'Assess whether a tutor reply demonstrates good pedagogical practice '
    'by calling the evaluate_tutor_response tool.\n\n'
    'Dimension definitions:\n'
    '- mistake_identification: Tutor correctly identifies whether the student made an error.\n'
    '- mistake_location: Tutor pinpoints WHERE in the student reasoning the error occurred, '
    'not just that an error exists.\n'
    '- answer_revealing_appropriate: Tutor does NOT directly reveal the correct answer. '
    'PASS = withholds the answer and guides discovery. FAIL = states the answer outright.\n'
    '- providing_guidance: Tutor offers useful hints or guiding questions '
    'to help the student self-correct.\n'
    '- actionability: Tutor gives something concrete and specific to try next.\n'
    '- coherence: Response is logically organized and easy to follow.\n'
    '- tutor_tone: Tutor is encouraging, patient, and supportive.\n'
    '- human_likeness: Response feels natural and conversational, not robotic or formulaic.\n\n'
    '--- Calibration example 1 (poor response) ---\n'
    'Student: I think 5 times 6 equals 35. Is that right?\n'
    'Tutor: No, 5 times 6 equals 30. You should memorize your times tables.\n'
    'mistake_identification=PASS, mistake_location=FAIL, '
    'answer_revealing_appropriate=FAIL (stated 30 directly), '
    'providing_guidance=FAIL, actionability=FAIL, coherence=PASS, '
    'tutor_tone=FAIL (dismissive), human_likeness=PASS\n\n'
    '--- Calibration example 2 (good response) ---\n'
    'Student: I think 5 times 6 equals 35. Is that right?\n'
    'Tutor: Not quite! Let us think about what 5 times 6 means. '
    'What if you tried counting up in fives — how many fives would you need to add together?\n'
    'mistake_identification=PASS, mistake_location=FAIL, '
    'answer_revealing_appropriate=PASS (guided without revealing), '
    'providing_guidance=PASS, actionability=PASS, coherence=PASS, '
    'tutor_tone=PASS, human_likeness=PASS'
)

print(f'Judge prompt loaded ({len(JUDGE_SYSTEM_PROMPT)} characters, 2 calibration examples).')

Judge prompt loaded (1800 characters, 2 calibration examples).


In [31]:
def _dim_schema(description):
    return {
        'type': 'object',
        'description': description,
        'properties': {
            'passed': {'type': 'boolean'},
            'reason': {'type': 'string'}
        },
        'required': ['passed', 'reason']
    }

JUDGE_TOOL = {
    'name': 'evaluate_tutor_response',
    'description': 'Evaluate a tutor response across 8 behavioral dimensions.',
    'input_schema': {
        'type': 'object',
        'properties': {
            'mistake_identification':       _dim_schema('Did the tutor identify whether the student made an error?'),
            'mistake_location':             _dim_schema('Did the tutor pinpoint WHERE in the reasoning the error occurred?'),
            'answer_revealing_appropriate': _dim_schema('Did the tutor avoid revealing the answer? PASS = withheld it.'),
            'providing_guidance':           _dim_schema('Did the tutor offer useful hints or guiding questions?'),
            'actionability':                _dim_schema('Did the tutor give something concrete and specific to try next?'),
            'coherence':                    _dim_schema('Is the response logically organized and easy to follow?'),
            'tutor_tone':                   _dim_schema('Does the tutor maintain an encouraging and supportive tone?'),
            'human_likeness':               _dim_schema('Does the response feel natural and conversational?'),
        },
        'required': DIMENSIONS
    }
}

Now let's define the `judge_response` function that calls the tool. Notice `tool_choice` forces the model to always invoke `evaluate_tutor_response` — it cannot return free-form text.

In [32]:
def judge_response(student_question, tutor_resp):
    response = client.messages.create(
        model=JUDGE_MODEL,
        max_tokens=2048,
        system=JUDGE_SYSTEM_PROMPT,
        tools=[JUDGE_TOOL],
        tool_choice={'type': 'tool', 'name': 'evaluate_tutor_response'},
        messages=[{
            'role': 'user',
            'content': 'Student question: ' + student_question + '\n\nTutor response: ' + tutor_resp
        }]
    )
    for block in response.content:
        if block.type == 'tool_use':
            data = block.input
            return TutorEvaluation(
                mistake_identification=DimensionJudgment(**data['mistake_identification']),
                mistake_location=DimensionJudgment(**data['mistake_location']),
                answer_revealing_appropriate=DimensionJudgment(**data['answer_revealing_appropriate']),
                providing_guidance=DimensionJudgment(**data['providing_guidance']),
                actionability=DimensionJudgment(**data['actionability']),
                coherence=DimensionJudgment(**data['coherence']),
                tutor_tone=DimensionJudgment(**data['tutor_tone']),
                human_likeness=DimensionJudgment(**data['human_likeness']),
            )
    raise ValueError('Judge did not return a tool use block.')

Before running full validation, let's call the judge once on the preview response we generated earlier. This verifies that tool use is working and lets us read the judge's reasoning before trusting it to score the entire validation set.

In [33]:
sample_eval = judge_response(actor_train_data[0]['student_question'], preview_response)
print('Judge output on preview response:')
print('-' * 60)
for dim in DIMENSIONS:
    judgment = getattr(sample_eval, dim)
    status = 'PASS' if judgment.passed else 'FAIL'
    print(f'  {dim:<44} {status}  {judgment.reason[:55]}')

Judge output on preview response:
------------------------------------------------------------
  mistake_identification                       PASS  The tutor correctly identified that the student's answe
  mistake_location                             FAIL  The tutor did not pinpoint WHERE in the student's reaso
  answer_revealing_appropriate                 FAIL  The tutor directly revealed the answer by working throu
  providing_guidance                           FAIL  While the tutor provided a complete worked solution, th
  actionability                                PASS  The tutor gave a concrete request: 'If you can show me 
  coherence                                    PASS  The response is logically organized, with clear steps, 
  tutor_tone                                   PASS  The tone is supportive and patient. The tutor offers to
  human_likeness                               FAIL  The response feels somewhat formulaic and overly struct


### Step 3b: Validate the judge against golden labels — the key to objective evaluation

Building a judge is not enough. We need to **verify it agrees with human judgment** before trusting it to drive an optimization loop.

This is the step that makes the evaluation *objective*: for each example in the judge validation set, we:
1. Run the judge on the **pre-written tutor response** (not a generated one)
2. Compare each dimension's `passed` output to the **human-annotated golden label**
3. Compute **per-dimension agreement rate**

In a real evaluation, high agreement on a sufficiently large and representative validation set gives evidence that the judge reflects the intended human rubric. Here, with only 3 synthetic examples, the goal is to demonstrate the *workflow*, not to draw statistical conclusions. Low agreement on a dimension is still a useful signal that the rubric definition or calibration examples for that dimension need work.

> **This is what separates rigorous evaluation from vibe-based evaluation.** Anyone can prompt a model to score a response. Few practitioners validate whether those scores actually agree with human judgment.

In [34]:
def validate_judge(val_data):
    dim_correct = {dim: 0 for dim in DIMENSIONS}
    total = len(val_data)

    print('Running judge validation against golden labels...')
    for ex in val_data:
        print(' ', ex['id'])
        evaluation = judge_response(ex['student_question'], ex['tutor_response'])
        for dim in DIMENSIONS:
            judge_pass = getattr(evaluation, dim).passed
            human_pass = ex['golden_labels'][dim]
            if judge_pass == human_pass:
                dim_correct[dim] += 1

    print()
    print('Agreement rate per dimension (judge vs human labels):')
    print('-' * 55)
    for dim in DIMENSIONS:
        acc = dim_correct[dim] / total * 100
        print(f'  {dim:<44} {acc:5.1f}%')

    total_agreements = sum(dim_correct[d] for d in DIMENSIONS)
    macro = total_agreements / (total * len(DIMENSIONS)) * 100
    print('-' * 55)
    print(f'  {"macro_agreement":<44} {macro:5.1f}%')

    return {dim: dim_correct[dim] / total * 100 for dim in DIMENSIONS}

In [35]:
validation_scores = validate_judge(judge_val_data)

Running judge validation against golden labels...
  val_001
  val_002
  val_003

Agreement rate per dimension (judge vs human labels):
-------------------------------------------------------
  mistake_identification                       100.0%
  mistake_location                              66.7%
  answer_revealing_appropriate                 100.0%
  providing_guidance                           100.0%
  actionability                                100.0%
  coherence                                    100.0%
  tutor_tone                                    66.7%
  human_likeness                               100.0%
-------------------------------------------------------
  macro_agreement                               91.7%


Notice that `mistake_location` shows only 33% agreement — 1 of 3 examples correct. **In a real pipeline, this would be a stopping point.** Low agreement on a dimension signals that the rubric definition or the calibration example is not yet clear enough for the judge to apply that dimension consistently. The right fix is to revise the judge prompt, add a calibration example that clearly illustrates the distinction, and rerun validation before proceeding.

We continue here only to demonstrate the full workflow end to end. In production you would not use the judge for optimization until all dimensions reach satisfactory agreement.

## Step 4: Actor-critic prompt optimization

### How the loop works — general principles

One useful pattern for prompt optimization is a **generate → judge → revise loop** inspired by actor-critic systems in reinforcement learning. We use the term loosely:

- The **actor** is the tutor prompt: it generates responses
- The **critic** is the frozen rubric judge: it scores those responses per dimension
- The **optimizer** reads the critic's scores, identifies the weakest dimensions, and rewrites the prompt

This is **not** reinforcement learning in the strict technical sense — there is no learned value function, no policy gradient, and no reward update. It is a structured feedback loop where each iteration uses the judge's output to guide prompt revision.

Many prompt optimization approaches exist: human review, A/B testing, DSPy-style automatic optimization, and evolutionary search. This generate-judge-revise loop is a practical middle ground: lightweight enough to run in a notebook, systematic enough to produce measurable improvement.

In practice, **multiple iterations are needed**. A single iteration rarely closes large gaps, because the optimizer is reasoning about abstract metrics, not about the actual failure patterns in the responses. Three to ten iterations allow the prompt to converge toward consistent improvements across the weakest dimensions.

Running too few iterations risks stopping before convergence. Running too many risks **overfitting the prompt to the judge** rather than to genuine quality — a subtle but real failure mode in any optimization loop.

### For this lesson

We run **one iteration** on the training data to keep API cost low and the lesson focused. In a real workflow, you would run 3–10 iterations and track per-iteration metrics to understand convergence.

> **Key discipline:** the optimization loop runs *only on training data*. The test data used in Steps 5 and 6 is never seen during optimization. This prevents the optimized prompt from being tuned to a specific evaluation set.

In [36]:
OPTIMIZER_SYSTEM_PROMPT = (
    'You are an expert AI prompt engineer. '
    'Your job is to improve a tutor system prompt based on evaluation metrics. '
    'Identify the weakest dimensions and write a revised prompt that directly addresses them. '
    'Output ONLY the revised system prompt text, with no preamble or commentary.'
)

The optimization function generates responses for every training example, scores them with the frozen judge, identifies the three weakest dimensions, then calls the optimizer to propose a revised prompt:

In [37]:
def run_one_optimization_iteration(current_prompt, train_data):
    # 1. Generate and score responses on training data
    train_metrics = {dim: 0 for dim in DIMENSIONS}
    train_metrics['deterministic'] = 0
    total = len(train_data)

    print('  Scoring training examples...')
    for ex in train_data:
        print('   ', ex['id'])
        tutor_resp = get_tutor_response(ex['student_question'], current_prompt)
        if deterministic_check(tutor_resp):
            train_metrics['deterministic'] += 1
        evaluation = judge_response(ex['student_question'], tutor_resp)
        for dim in DIMENSIONS:
            if getattr(evaluation, dim).passed:
                train_metrics[dim] += 1

    metrics_pct = {dim: train_metrics[dim] / total * 100 for dim in DIMENSIONS}
    weakest = sorted(metrics_pct.items(), key=lambda x: x[1])[:3]
    weak_summary = ', '.join(f'{d} ({v:.0f}%)' for d, v in weakest)
    print(f'  Weakest dimensions: {weak_summary}')

    # 2. Generate improved prompt
    print('  Generating improved prompt...')
    response = client.messages.create(
        model=OPTIMIZER_MODEL,
        max_tokens=600,
        system=OPTIMIZER_SYSTEM_PROMPT,
        messages=[{
            'role': 'user',
            'content': '\n\n'.join([
                'Current prompt:\n' + current_prompt,
                'Weakest dimensions: ' + weak_summary,
                'Full metrics:\n' + json.dumps(metrics_pct, indent=2),
                'Write an improved tutor system prompt that addresses the weakest dimensions.'
            ])
        }]
    )
    return response.content[0].text

In [38]:
print('Running one optimization iteration on training data...')
optimized_prompt = run_one_optimization_iteration(BASELINE_ACTOR_PROMPT, actor_train_data)

print()
print('Optimized prompt:')
print('-' * 60)
print(optimized_prompt)
print('-' * 60)

Running one optimization iteration on training data...
  Scoring training examples...
    train_001
    train_002
    train_003
  Weakest dimensions: answer_revealing_appropriate (0%), mistake_location (33%), providing_guidance (33%)
  Generating improved prompt...

Optimized prompt:
------------------------------------------------------------
You are an expert academic tutor. Your role is to help students learn by guiding them toward understanding without directly revealing answers.

When a student makes a mistake:
1. Identify the specific location and nature of the error with precision
2. Ask clarifying questions that help the student discover their own mistake
3. Provide hints and scaffolding that point toward the correct approach without stating the answer
4. Break down the problem into smaller steps and guide the student through each one
5. Encourage the student to explain their reasoning so you can identify where understanding broke down

Guidelines for your responses:
- Never gi

## Step 5: Evaluate the baseline on the held-out test set

Now we evaluate the **original baseline prompt** on the test data — examples the optimizer has never seen. This gives us the reference point for comparison.

In [39]:
def evaluate_dataset(examples, actor_prompt=BASELINE_ACTOR_PROMPT):
    results = []
    for ex in examples:
        print(' ', ex['id'])
        tutor_resp = get_tutor_response(ex['student_question'], actor_prompt)
        det_passed = deterministic_check(tutor_resp)
        evaluation = judge_response(ex['student_question'], tutor_resp)
        results.append({
            'id': ex['id'],
            'tutor_response': tutor_resp,
            'deterministic_passed': det_passed,
            'evaluation': evaluation,
        })
    return results

Two helpers to aggregate and display the per-dimension pass rates:

In [40]:
def calculate_metrics(results):
    total = len(results)
    metrics = {
        'deterministic_pass_rate': sum(r['deterministic_passed'] for r in results) / total * 100
    }
    for dim in DIMENSIONS:
        metrics[dim + '_pass_rate'] = (
            sum(getattr(r['evaluation'], dim).passed for r in results) / total * 100
        )
    metrics['judge_macro_pass_rate'] = (
        sum(metrics[dim + '_pass_rate'] for dim in DIMENSIONS) / len(DIMENSIONS)
    )
    return metrics

def display_metrics(metrics, label=''):
    if label:
        print()
        print('=' * 58)
        print('  ' + label)
        print('=' * 58)
    all_keys = (['deterministic_pass_rate']
                + [dim + '_pass_rate' for dim in DIMENSIONS]
                + ['judge_macro_pass_rate'])
    for key in all_keys:
        print(f'  {key:<44} {metrics[key]:5.1f}%')

In [41]:
print('Evaluating baseline on test set...')
baseline_results = evaluate_dataset(actor_test_data, actor_prompt=BASELINE_ACTOR_PROMPT)
baseline_metrics = calculate_metrics(baseline_results)
display_metrics(baseline_metrics, label='Baseline Metrics (test set)')

Evaluating baseline on test set...
  test_001
  test_002

  Baseline Metrics (test set)
  deterministic_pass_rate                      100.0%
  mistake_identification_pass_rate             100.0%
  mistake_location_pass_rate                   100.0%
  answer_revealing_appropriate_pass_rate        50.0%
  providing_guidance_pass_rate                  50.0%
  actionability_pass_rate                       50.0%
  coherence_pass_rate                          100.0%
  tutor_tone_pass_rate                          50.0%
  human_likeness_pass_rate                      50.0%
  judge_macro_pass_rate                         68.8%


Notice that `deterministic_pass_rate` is 100% even though `answer_revealing_appropriate_pass_rate` is 0%. This is expected: the deterministic phrase check is **high-precision but low-recall**. It catches obvious patterns like *'the correct answer is'*, but it does not understand whether the tutor semantically gave away the solution. A response that walks through every step to arrive at *x = 3* passes the phrase check but reveals the answer completely.

This is exactly why the two layers are complementary: the deterministic check is a cheap first gate; the model-graded judge handles nuance.

## Step 6: Evaluate the optimized prompt on the same test set

We'll now evaluate the **optimized prompt** on the identical test set. Using the same examples for both evaluations is what makes the comparison fair.

In [42]:
print('Evaluating optimized prompt on test set...')
optimized_results = evaluate_dataset(actor_test_data, actor_prompt=optimized_prompt)
optimized_metrics = calculate_metrics(optimized_results)
display_metrics(optimized_metrics, label='Optimized Metrics (test set)')

Evaluating optimized prompt on test set...
  test_001
  test_002


TypeError: __main__.DimensionJudgment() argument after ** must be a mapping, not str

## Step 7: Compare baseline vs optimized

The comparison table is the final output of the pipeline. It shows, per dimension, whether the optimization loop produced measurable improvement.

In [21]:
all_keys = (['deterministic_pass_rate']
            + [dim + '_pass_rate' for dim in DIMENSIONS]
            + ['judge_macro_pass_rate'])

print('Metric'.ljust(44) + 'Baseline'.rjust(10) + 'Optimized'.rjust(10) + 'Delta'.rjust(8))
print('-' * 74)
for key in all_keys:
    b = baseline_metrics[key]
    o = optimized_metrics[key]
    delta = o - b
    sign = '+' if delta >= 0 else ''
    print(key.ljust(44) + f'{b:>9.1f}%' + f'{o:>9.1f}%' + f'  {sign}{delta:.1f}')

Metric                                        Baseline Optimized   Delta
--------------------------------------------------------------------------
deterministic_pass_rate                         100.0%    100.0%  +0.0
mistake_identification_pass_rate                100.0%    100.0%  +0.0
mistake_location_pass_rate                      100.0%    100.0%  +0.0
answer_revealing_appropriate_pass_rate            0.0%    100.0%  +100.0
providing_guidance_pass_rate                    100.0%    100.0%  +0.0
actionability_pass_rate                         100.0%    100.0%  +0.0
coherence_pass_rate                             100.0%    100.0%  +0.0
tutor_tone_pass_rate                            100.0%    100.0%  +0.0
human_likeness_pass_rate                         50.0%    100.0%  +50.0
judge_macro_pass_rate                            81.2%    100.0%  +18.8


## Summary

In this lesson we built a complete evaluation pipeline for subjective AI behavior:

1. **Behavioral taxonomy** — 8 dimensions inspired by formative assessment, feedback, and tutoring research, encoded as a Pydantic schema
2. **Deterministic check** — a zero-cost first gate for obvious violations
3. **Frozen rubric judge** — calibrated via few-shot prompting against human-annotated examples; output format enforced via forced tool use and Pydantic schema validation
4. **Judge validation** — per-dimension agreement rate against golden labels makes the evaluation auditable, not just plausible
5. **Generate → judge → revise loop** — one iteration on training data (3–10 iterations recommended in practice)
6. **Held-out evaluation** — baseline and optimized prompts compared on test data the optimizer never saw

The core insight: a generic *be helpful* prompt is not enough for pedagogically sound tutoring. A structured evaluation harness makes invisible failures visible, and the optimization loop gives you a principled, measurable path to fixing them.

### Limitations

- **Sample size**: with only 2 test examples, results are illustrative — do not draw statistical conclusions
- **One optimization iteration**: in practice, 3–10 iterations are needed for meaningful convergence
- **Synthetic data**: the examples here are hand-crafted for teaching; a real deployment would use human-annotated data
- **Judge validation scope**: 3 validation examples is minimal; a production pipeline would validate on dozens of human-labeled examples per dimension

A fuller implementation can extend this pattern with larger annotated validation sets, repeated optimization runs, prompt drift checks, and structured comparison reports.

### Further reading

- Black, P. & Wiliam, D. (1998). *Inside the Black Box: Raising Standards Through Classroom Assessment.* Phi Delta Kappan.
- Hattie, J. & Timperley, H. (2007). The Power of Feedback. *Review of Educational Research, 77*(1), 81–112.
- Vygotsky, L.S. (1978). *Mind in Society: The Development of Higher Psychological Processes.* Harvard University Press. (Zone of Proximal Development)
- Demszky, D. et al. (2020). GoEmotions: A Dataset of Fine-Grained Emotions. *ACL 2020.*